# Theorem 3 — diffusion–flow marginal equivalence

**Formal source:** [`../03_diffusion_flow_marginal_equivalence.md`](../03_diffusion_flow_marginal_equivalence.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(3)
count, steps, diffusion = 25000, 200, 0.3
dt = 1 / steps
initial = rng.normal(size=count)
flow = initial.copy()
stochastic = initial.copy()
for step in range(steps):
    time = step * dt
    mean, mean_dot = 0.4 * time, 0.4
    sigma, sigma_dot = 1 + 0.2 * time, 0.2
    flow += (mean_dot + sigma_dot / sigma * (flow - mean)) * dt
    score = -(stochastic - mean) / sigma ** 2
    stochastic += (mean_dot + sigma_dot / sigma * (stochastic - mean) + diffusion * score) * dt
    stochastic += math.sqrt(2 * diffusion * dt) * rng.normal(size=count)
assert abs(flow.mean() - 0.4) < 0.02 and abs(flow.var() - 1.44) < 0.04
assert abs(stochastic.mean() - 0.4) < 0.03 and abs(stochastic.var() - 1.44) < 0.05
assert np.mean(abs(flow - stochastic)) > 0.1
print({"flow_mean": float(flow.mean()), "sde_mean": float(stochastic.mean()), "flow_var": float(flow.var()), "sde_var": float(stochastic.var())})

In [ ]:
print('THEORY_DEMO_PASS::03_diffusion_flow_marginal_equivalence')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')